# Week 2 — Graph Laplacian diffusion

## Goal

Replace manual neighbour averaging with a mathematically clearer diffusion model.

## Why use a Laplacian?

Week 1 already behaved like diffusion, but the rule was ad hoc.

A more principled approach is:
- build adjacency matrix `A`
- degree matrix `D`
- Laplacian `L = D - A`

Then model signal transport using `L @ u`.

In [ ]:
import numpy as np

def line_adjacency(n: int) -> np.ndarray:
    A = np.zeros((n, n), dtype=float)
    for i in range(n - 1):
        A[i, i + 1] = 1.0
        A[i + 1, i] = 1.0
    return A

def simulate_laplacian_line(
    n_nodes: int = 20,
    corrected_fraction: float = 0.1,
    steps: int = 100,
    alpha: float = 0.4,
    beta: float = 0.05,
    dt: float = 0.1,
    threshold: float = 1.0,
):
    A = line_adjacency(n_nodes)
    D = np.diag(A.sum(axis=1))
    L = D - A

    u = np.zeros(n_nodes, dtype=float)
    rescued = np.zeros(n_nodes, dtype=bool)

    n_corrected = max(1, round(n_nodes * corrected_fraction))
    corrected = np.zeros(n_nodes, dtype=bool)
    corrected[:n_corrected] = True
    q = corrected.astype(float) * 0.5

    for _ in range(steps):
        du = -alpha * (L @ u) - beta * u + q
        u = np.maximum(u + dt * du, 0.0)
        rescued = np.logical_or(rescued, u >= threshold)

    return {"signal": u, "rescued": rescued, "adjacency": A, "laplacian": L}

## Intuition behind the equation

`-alpha * (L @ u)` spreads signal across neighbours.

`-beta * u` makes signal decay explicitly.

`+ q` keeps corrected nuclei as continuous sources.

So the model is now:

**transport + decay + production**

In [ ]:
result = simulate_laplacian_line()
result["signal"], result["rescued"]